# Colab classification of statistics descriptions 

## Set-up

In [ ]:
!pip install -U datasets
!pip install ray

In [ ]:
import datasets
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import random
import numpy as np
from datasets import Dataset

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Classification parameters
# Clear PyTorch GPU cache
torch.cuda.empty_cache()

# Training set parameters
fraction_positives = 1
fraction_mining = 0.1
ratio_factor_negatives = 1            # share of negatives: n_negatives = 1.5 * n_positives
fraction_positives_in_test = 0.05     # percent positives in test set
fraction_remaining_unlabeled = 1      # percent to keep of unlabeled for reconstructing training set

# Model parameters
model_id = "xlm-roberta-large"
max_token_length = None               # None uses max model allows

# Hyperparameter tuning
fraction_positives_for_tuning = 0.3
n_trials_ray = 30
n_cpu_ray = 2
n_gpu_ray = 1                         # For colab, 1 GPU max

# Prediction parameters
fraction_unlabeled_for_prediction = 0.1

In [ ]:
# Mount drive to import datasets
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Load the data
stat_to_mine = pd.read_feather("./drive/MyDrive/Colab Notebooks/stat_to_mine.feather")

# Discard any missing values in text_mining_description for robustness
stat_to_mine = stat_to_mine.dropna(subset=['text_mining_description'])

## Initial classification

This step was only done initially to create a first training set from `stat_to_mine` by stratified sampling and including minig descriptions as negatives. Later, this initial training set was adjusted manually by removing false positives and including desired descriptions (e.g. related to birth registration) to guide the model during training to capute a desired concept of data & statistics.

### Construct training and test set 

In [ ]:
# Filter out positive, mining and unlabeled samples
positive = stat_to_mine[stat_to_mine['is_statistics'] == True].copy()
mining = stat_to_mine[stat_to_mine['is_mining'] == True].copy()
unlabeled = stat_to_mine[(stat_to_mine['is_statistics'] == False) & (stat_to_mine['is_mining'] == False)].copy()

# Sample randomly fraction_positives for positives
positive = positive.sample(frac=fraction_positives, random_state=SEED)

In [ ]:
# --------------------------  Mining  ------------------------------------------
# Sample fraction_mining of mining (scale the number of mining samples with fraction_positives so that the ratio is maintained)
n_mining = int(fraction_positives * fraction_mining * len(mining))
mining_sampled = mining.sample(n=n_mining, random_state=SEED)

# ---------------- Stratified sampling from unlabeled --------------------------
# Stratification acc. to language and length of text_mining_description
# Create length bins according to language and length of text_mining_description
positive.loc[:, 'length_bin'] = pd.cut(positive['text_mining_description'].str.len(), bins=6, labels=False, include_lowest=True)
unlabeled.loc[:, 'length_bin'] = pd.cut(unlabeled['text_mining_description'].str.len(), bins=6, labels=False, include_lowest=True)
mining_sampled.loc[:, 'length_bin'] = pd.cut(mining_sampled['text_mining_description'].str.len(), bins=6, labels=False, include_lowest=True)

# Create joint stratum
positive.loc[:, 'stratum'] = positive['language'].astype(str) + "_" + positive['length_bin'].astype(str)
unlabeled.loc[:, 'stratum'] = unlabeled['language'].astype(str) + "_" + unlabeled['length_bin'].astype(str)
mining_sampled.loc[:, 'stratum'] = mining_sampled['language'].astype(str) + "_" + mining_sampled['length_bin'].astype(str)

# Calculate stratum distribution in positives
stratum_counts = positive['stratum'].value_counts(normalize=True)

# Sample from unlabeled according to this distribution
n_negatives = ratio_factor_negatives*len(positive)-n_mining
samples_per_stratum = (stratum_counts * n_negatives).round().astype(int)

# Sample from unlabeled
neg_samples = []
for stratum, n in samples_per_stratum.items():
    candidates = unlabeled[unlabeled['stratum'] == stratum]
    if len(candidates) >= n:
        neg_samples.append(candidates.sample(n=n, random_state=SEED))
    else:
        neg_samples.append(candidates)  # take all if not enough

unlabeled_sampled = pd.concat(neg_samples).sample(frac=1, random_state=SEED)  # shuffle

# Combine positives and sampled negatives
balanced = pd.concat([positive, unlabeled_sampled, mining_sampled]).reset_index(drop=True)

# Keep the remaining unlabeled samples for later
remaining_unlabeled = unlabeled.drop(unlabeled_sampled.index)

In [ ]:
# Split into train & test sets (80% train, 20% test)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    balanced['text_mining_description'],
    balanced['is_statistics'].astype(int),
    test_size=0.1,
    stratify=balanced['is_statistics'],
    random_state=SEED
)

balanced['is_statistics'].value_counts()

Optional: unbalance test set to resemble real world distribution of stat projects in CRS, but was later abbandoned in favor of a manual inspection of descriptions around the eventual threshold

In [ ]:
# Unbalance test set to resemble real world distribution
n_pos_test = sum(test_labels)
n_neg_test = len(test_labels) - n_pos_test
n_neg_add = int((n_pos_test - fraction_positives_in_test*(n_pos_test + n_neg_test))/fraction_positives_in_test)

print(n_neg_add)

additional_unlabeled = remaining_unlabeled.sample(n=n_neg_add, random_state=SEED)

additional_unlabeled_texts  = additional_unlabeled['text_mining_description']
additional_unlabeled_labels = additional_unlabeled['is_statistics'].astype(int)

additional_unlabeled_labels.value_counts()

In [ ]:
test_texts = pd.concat([test_texts, additional_unlabeled_texts])
test_labels =  pd.concat([test_labels, additional_unlabeled_labels])

print(f"Final‑test prevalence: {np.mean(test_labels)}")

### Tokenization

In [ ]:
# Load tokenizer and tokenize data
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=max_token_length)

# Create Hugging Face datasets
train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'label': train_labels.tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'label': test_labels.tolist()})

# Tokenize dataset with custom tokenize()
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

### Model training

In [ ]:
# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./drive/MyDrive/Colab Notebooks/models/results",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=1e-5,
    logging_dir="./drive/MyDrive/Colab Notebooks/models/logs",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1, # keeps only the latest checkpoint
    metric_for_best_model="precision",
    fp16=True,
    report_to="none"
)

# Define metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

In [ ]:
predictions_output = trainer.predict(test_dataset)

In [ ]:
# Output test set with, text, label, and predicted probabilities
test_results = pd.DataFrame({
    'text': test_texts,
    'label': test_labels,
    'probability_is_statistics': torch.softmax(torch.tensor(predictions_output.predictions), dim=1)[:, 1].numpy()
})

#test_results.to_excel("./drive/MyDrive/Colab Notebooks/test_set_predicted.xlsx", index=False)

### Model evaluation

In [ ]:
# Interactive ROC curve
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score
import plotly.graph_objs as go
import matplotlib.pyplot as plt

# Extract labels and predicted probabilities
y_true = test_results['label'].values
y_scores = test_results['probability_is_statistics'].values

# Calculate ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = roc_auc_score(y_true, y_scores)

# Build interactive ROC plot
hover_text = [f"Threshold: {thr:.3f}<br>FPR: {f:.3f}<br>TPR: {t:.3f}"
              for thr, f, t in zip(thresholds, fpr, tpr)]

trace = go.Scatter(
    x=fpr,
    y=tpr,
    mode='lines+markers',
    text=hover_text,
    hoverinfo='text',
    name=f'ROC curve (AUC = {roc_auc:.3f})',
    line=dict(color='orange')
)

line_random = go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Guessing',
    line=dict(dash='dash', color='navy')
)

layout = go.Layout(
    title='ROC Curve',
    xaxis=dict(title='False Positive Rate'),
    yaxis=dict(title='True Positive Rate (Recall)'),
    legend=dict(x=0.6, y=0.05),
    hovermode='closest'
)

fig = go.Figure(data=[trace, line_random], layout=layout)
fig.show()


In [ ]:
# Precision recall curve
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, pr_thresholds = precision_recall_curve(y_true, y_scores)
pr_auc = average_precision_score(y_true, y_scores)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'PR curve (AP = {pr_auc:.3f})', color='green', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall Curve on Unbalanced Test Set')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

## Rebuild training set with reliable negatives

### Predict unlabeled

Predict the unlabeled set to obtain reliable negatives for reconstructing the training set later on

In [ ]:
# Sample from remaining unlabeled
remaining_unlabeled = remaining_unlabeled.sample(frac=fraction_remaining_unlabeled, random_state=SEED)

# Construct dataset 
remaining_unlabeled_texts = remaining_unlabeled["text_mining_description"].tolist()
remaining_unlabeled_dataset = Dataset.from_dict({'text': remaining_unlabeled_texts})
remaining_unlabeled_dataset = remaining_unlabeled_dataset.map(tokenize, batched=True)

In [ ]:
# Predict the unlabeled samples
pred_output = trainer.predict(remaining_unlabeled_dataset)
logits = pred_output.predictions
probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
remaining_unlabeled["probability_is_statistics"] = probs

In [ ]:
# Save to xlsx to inspect manually
remaining_unlabeled.to_excel("./drive/MyDrive/Colab Notebooks/unlabeled_predictions_1training_large.xlsx", index=False)

### Rebuild training set with reliable negatives

In [ ]:
# --------------------------- Positives ----------------------------------------
# Filter out positive and unlabeled samples
positive = stat_to_mine[stat_to_mine['is_statistics'] == True].copy()

# Sample randomly fraction_positives for positives
positive = positive.sample(frac=fraction_positives_for_tuning, random_state=SEED)

# Use the same stratum distribution as positives
positive.loc[:, 'length_bin'] = pd.cut(
    positive['text_mining_description'].str.len(), bins=6, labels=False, include_lowest=True
)
positive.loc[:, 'stratum'] = positive['language'].astype(str) + "_" + positive['length_bin'].astype(str)
stratum_counts = positive['stratum'].value_counts(normalize=True)

# ------------------------------- Mining ---------------------------------------
mining = stat_to_mine[stat_to_mine['is_mining'] == True].copy()
n_mining = int(fraction_positives_for_tuning * fraction_mining * len(mining))
mining_sampled = mining.sample(n=n_mining, random_state=SEED)


# ------------------------- Reliable negatives ---------------------------------
# Select reliable negatives: probability < 0.8 (stratum & language still present from earlier), 0.8 chosen to ensure that we capture a wide range of examples so that some near statistics projects are included 
reliable_negatives = remaining_unlabeled[remaining_unlabeled['probability_is_statistics'] < 0.8].copy()

n_negatives = len(positive)-n_mining
samples_per_stratum = (stratum_counts * n_negatives).round().astype(int)

neg_samples = []
for stratum, n in samples_per_stratum.items():
    candidates = reliable_negatives[reliable_negatives['stratum'] == stratum]
    if len(candidates) >= n:
        neg_samples.append(candidates.sample(n=n, random_state=SEED))
    else:
        neg_samples.append(candidates)

reliable_negatives_sampled = pd.concat(neg_samples).sample(frac=1, random_state=SEED)
reliable_negatives_sampled = reliable_negatives_sampled.drop(columns=['probability_is_statistics', 'length_bin', 'stratum'])

# Rebuild the training set
positive = positive.drop(columns=['length_bin', 'stratum'])
training_dataset_with_reliables = pd.concat([positive, mining_sampled, reliable_negatives_sampled]).reset_index(drop=True)
training_dataset_with_reliables = training_dataset_with_reliables.drop(columns=['is_mining'], errors='ignore')

training_dataset_with_reliables['is_statistics'].value_counts()